In [ ]:
import os
import numpy as np
import torch
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

In [ ]:
base_dir = r"" # path to the data

image_slices = []
label_slices = []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue

    patient_num = folder_name.split("-")[1]
    flair_path  = os.path.join(root, f"{patient_num}-Flair.nii")
    label_path  = os.path.join(root, f"{patient_num}-LesionSeg-Flair.nii")

    if not (os.path.exists(flair_path) and os.path.exists(label_path)):
        continue

    # Load NIfTI volumes
    flair_vol = nib.load(flair_path).get_fdata() 
    label_vol = nib.load(label_path).get_fdata()

    # Normalize intensity to [0, 1]
    flair_vol = np.clip(flair_vol, 0, 1000)
    flair_vol = flair_vol / 1000.0

    # Binarize label
    label_vol = (label_vol > 0).astype(np.float32)

    # Extract each slice along z-axis
    num_slices = flair_vol.shape[2]
    for z in range(num_slices):
        image_slices.append(flair_vol[:, :, z].astype(np.float32))
        label_slices.append(label_vol[:, :, z].astype(np.float32))

print(f"Total 2D slices extracted : {len(image_slices)}")
print(f"Sample slice shape        : {image_slices[0].shape}")
print(f"Label unique values       : {np.unique(label_slices[0])}")

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Install albumentations if not present
try:
    import albumentations
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", 
                   "install", "albumentations"], check=True)
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

TARGET_SIZE = 256   # resize all slices to 256x256

train_aug = A.Compose([
    A.Resize(TARGET_SIZE, TARGET_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, 
                       rotate_limit=15, p=0.5),
    A.ElasticTransform(p=0.3),
    A.GaussianBlur(p=0.2),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(TARGET_SIZE, TARGET_SIZE),
    ToTensorV2(),
])

class SliceDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]   # (H, W) float32
        label = self.labels[idx]   # (H, W) float32

        if self.transform:
            aug   = self.transform(image=image, mask=label)
            image = aug["image"].float()    # (1, H, W)
            label = aug["mask"].float()     # (H, W)
        
        # Add channel dim to image if needed
        if image.dim() == 2:
            image = image.unsqueeze(0)

        return image, label

print("Dataset class defined.")

In [ ]:
base_dir = r"" # path to the data


slice_counts  = []
patient_nums  = []

for root, dirs, files in os.walk(base_dir):
    folder_name = os.path.basename(root)
    if not folder_name.startswith("Patient-"):
        continue
    patient_num = folder_name.split("-")[1]
    flair_path  = os.path.join(root, f"{patient_num}-Flair.nii")
    if os.path.exists(flair_path):
        vol = nib.load(flair_path).get_fdata()
        slice_counts.append(vol.shape[2])
        patient_nums.append(int(patient_num))


paired       = sorted(zip(patient_nums, slice_counts))
slice_counts = [s for _, s in paired]

total_patients  = len(slice_counts)
split_idx       = int(0.8 * total_patients)   
train_slice_end = sum(slice_counts[:split_idx])

train_images = image_slices[:train_slice_end]
train_labels = label_slices[:train_slice_end]
val_images   = image_slices[train_slice_end:]
val_labels   = label_slices[train_slice_end:]

print(f"Total patients   : {total_patients}")
print(f"Train patients   : {split_idx}")
print(f"Val patients     : {total_patients - split_idx}")
print(f"Train slices     : {len(train_images)}")
print(f"Val slices       : {len(val_images)}")


filtered_train_images = []
filtered_train_labels = []
kept    = 0
skipped = 0

for img, lbl in zip(train_images, train_labels):
    if lbl.max() > 0:  
        filtered_train_images.append(img)
        filtered_train_labels.append(lbl)
        kept += 1
    else:
        skipped += 1

print(f"\nTrain slices with lesion    : {kept}")
print(f"Train slices without lesion : {skipped} (removed)")
print(f"Val slices kept as-is       : {len(val_images)}")

train_ds     = SliceDataset(filtered_train_images, filtered_train_labels,
                            transform=train_aug)
val_ds       = SliceDataset(val_images, val_labels, transform=val_aug)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from monai.losses import DiceLoss

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1,
                 features=[32, 64, 128, 256]):
        super().__init__()
        self.encoders   = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        self.pool       = nn.MaxPool2d(2)

        
        ch = in_channels
        for f in features:
            self.encoders.append(ConvBlock(ch, f))
            ch = f

        
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        
        for f in reversed(features):
            self.decoders.append(nn.ConvTranspose2d(f * 2, f, 2, stride=2))
            self.decoders.append(ConvBlock(f * 2, f))

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skip_connections = []

        for encoder in self.encoders:
            x = encoder(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.decoders), 2):
            x    = self.decoders[i](x)
            skip = skip_connections[i // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decoders[i + 1](x)

        return self.final(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")
if device.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")


model = UNet2D(in_channels=1, out_channels=1).to(device)

# ── Loss — combined Dice + weighted BCE for 345:1 imbalance ───
dice_loss_fn = DiceLoss(sigmoid=True)
bce_loss_fn  = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([345.0]).to(device)
)

def combined_loss(outputs, labels):
    return 0.5 * bce_loss_fn(outputs, labels) + \
           0.5 * dice_loss_fn(outputs, labels)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=10,
    min_lr=1e-7,
    verbose=True,
)

print(f"Trainable parameters : "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Model, loss, optimizer ready.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def dice_coefficient(pred, target, threshold=0.5, smooth=1e-6):
    pred   = (torch.sigmoid(pred) > threshold).float()
    inter  = (pred * target).sum()
    return ((2 * inter + smooth) / (pred.sum() + target.sum() + smooth)).item()

RESUME_FROM  = None
max_epochs   = 150
best_val_dice = 0.0
best_epoch   = 0
train_loss_log = []
val_dice_log   = []
start_epoch  = 1
PATIENCE     = 20
no_improve   = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    checkpoint     = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    best_val_dice  = checkpoint["best_val_dice"]
    train_loss_log = checkpoint.get("train_loss_log", [])
    val_dice_log   = checkpoint.get("val_dice_log", [])
    start_epoch    = checkpoint["epoch"] + 1
    print(f"Resumed from epoch {checkpoint['epoch']} | "
          f"Best Dice: {best_val_dice:.4f}")
else:
    print("Starting training from scratch.")

for epoch in range(start_epoch, max_epochs + 1):

    # ── Train ──
    model.train()
    epoch_loss  = 0.0
    epoch_steps = 0

    with tqdm(train_loader, desc=f"Epoch {epoch:>3}/{max_epochs} [Train]",
              leave=False) as pbar:
        for images_b, labels_b in pbar:
            images_b = images_b.to(device)
            labels_b = labels_b.to(device).unsqueeze(1)  # (B,1,H,W)

            optimizer.zero_grad()
            outputs = model(images_b)
            loss = combined_loss(outputs, labels_b)
            loss.backward()
            optimizer.step()

            epoch_loss  += loss.item()
            epoch_steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = epoch_loss / epoch_steps
    train_loss_log.append(avg_train_loss)


    if epoch % 2 == 0:
        model.eval()
        torch.cuda.empty_cache()
        val_dices = []

        with torch.no_grad():
            for images_b, labels_b in val_loader:
                images_b = images_b.to(device)
                labels_b = labels_b.to(device).unsqueeze(1)
                outputs  = model(images_b)
                dice     = dice_coefficient(outputs, labels_b)
                val_dices.append(dice)

        mean_dice = np.mean(val_dices)
        val_dice_log.append((epoch, mean_dice))
        scheduler.step(mean_dice)

        print(f"Epoch [{epoch:>3}/{max_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}  |  "
              f"Val Dice: {mean_dice:.4f}")

        if mean_dice > best_val_dice:
            best_val_dice = mean_dice
            best_epoch    = epoch
            no_improve    = 0
            torch.save(model.state_dict(), "best_unet2d.pth")
            print(f"  ✓ New best model saved (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve} val checks "
                  f"(patience: {PATIENCE})")
            if no_improve >= PATIENCE:
                print(f"\n⚠ Early stopping at epoch {epoch}")
                break
    else:
        print(f"Epoch [{epoch:>3}/{max_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}")

    if epoch % 10 == 0:
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_dice":        best_val_dice,
            "train_loss_log":       train_loss_log,
            "val_dice_log":         val_dice_log,
        }, f"unet2d_checkpoint_epoch_{epoch}.pth")
        print(f"  💾 Checkpoint saved at epoch {epoch}")

    torch.cuda.empty_cache()

print(f"\nTraining complete.")
print(f"Best Val Dice: {best_val_dice:.4f} at epoch {best_epoch}")

In [ ]:
import matplotlib.pyplot as plt

epochs_logged = list(range(1, len(train_loss_log) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_logged, train_loss_log, color="steelblue", linewidth=1)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

if val_dice_log:
    val_epochs = [x[0] for x in val_dice_log]
    val_dices  = [x[1] for x in val_dice_log]
    axes[1].plot(val_epochs, val_dices, color="darkorange",
                 linewidth=1, marker="o", markersize=3)
    axes[1].set_title("Validation Dice Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves_2d.png", dpi=150)
plt.show()

print(f"Best val Dice  : {max(val_dice_log, key=lambda x: x[1])}")
print(f"Current LR     : {optimizer.param_groups[0]['lr']}")

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch


model.load_state_dict(torch.load("best_unet2d.pth", map_location=device))
model.eval()


lesion_val_indices = [i for i, lbl in enumerate(val_labels) if lbl.max() > 0]
print(f"Val slices with lesions: {len(lesion_val_indices)}")

viz_indices = lesion_val_indices[::max(1, len(lesion_val_indices)//6)][:6]

fig, axes = plt.subplots(len(viz_indices), 4, figsize=(16, 4 * len(viz_indices)))
fig.suptitle("Lesion Segmentation — FLAIR MRI\nModel Predictions vs Ground Truth",
             fontsize=14, fontweight="bold", y=1.01)

col_titles = ["Input FLAIR", "Ground Truth", "Prediction", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight="bold")

with torch.no_grad():
    for row, idx in enumerate(viz_indices):
        image = val_images[idx]   
        label = val_labels[idx]   

        
        from albumentations.pytorch import ToTensorV2
        import albumentations as A
        transform = A.Compose([A.Resize(256, 256), ToTensorV2()])
        aug       = transform(image=image, mask=label)
        img_t = aug["image"].float().unsqueeze(0).to(device)
        lbl_t     = aug["mask"].numpy()

       
        output    = model(img_t)
        pred      = (torch.sigmoid(output) > 0.5).float()
        pred_np   = pred.squeeze().cpu().numpy()
        img_np    = img_t.squeeze().cpu().numpy()

        
        inter     = (pred_np * lbl_t).sum()
        dice      = (2 * inter + 1e-6) / (pred_np.sum() + lbl_t.sum() + 1e-6)

        
        axes[row, 0].imshow(img_np, cmap="gray")
        axes[row, 0].set_ylabel(f"Slice {idx}\nDice: {dice:.3f}",
                                fontsize=9)
        axes[row, 0].axis("off")

        
        axes[row, 1].imshow(img_np, cmap="gray")
        axes[row, 1].imshow(np.ma.masked_where(lbl_t == 0, lbl_t),
                            cmap="Greens", alpha=0.6, vmin=0, vmax=1)
        axes[row, 1].axis("off")

        
        axes[row, 2].imshow(img_np, cmap="gray")
        axes[row, 2].imshow(np.ma.masked_where(pred_np == 0, pred_np),
                            cmap="Reds", alpha=0.6, vmin=0, vmax=1)
        axes[row, 2].axis("off")

        
        axes[row, 3].imshow(img_np, cmap="gray")
        
        tp = np.ma.masked_where((lbl_t == 0) | (pred_np == 0),
                                np.ones_like(lbl_t))
        
        fn = np.ma.masked_where((lbl_t == 0) | (pred_np == 1),
                                np.ones_like(lbl_t))
        
        fp = np.ma.masked_where((lbl_t == 1) | (pred_np == 0),
                                np.ones_like(lbl_t))

        axes[row, 3].imshow(tp, cmap="autumn",  alpha=0.7, vmin=0, vmax=1)
        axes[row, 3].imshow(fn, cmap="Greens",  alpha=0.6, vmin=0, vmax=1)
        axes[row, 3].imshow(fp, cmap="cool",    alpha=0.5, vmin=0, vmax=1)
        axes[row, 3].axis("off")


tp_patch = mpatches.Patch(color="yellow",      label="True Positive")
fn_patch = mpatches.Patch(color="green",       label="False Negative (missed)")
fp_patch = mpatches.Patch(color="cyan",        label="False Positive (extra)")
fig.legend(handles=[tp_patch, fn_patch, fp_patch],
           loc="lower center", ncol=3, fontsize=10,
           bbox_to_anchor=(0.75, -0.02))

plt.tight_layout()
plt.savefig("proposal_predictions.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved as proposal_predictions.png")